In [ ]:
import sys
import os

# Add the repo root to path so 'src' is findable
sys.path.insert(0, os.path.abspath('./Barak-Cajiao-Trabecular-Bone-Analysis'))


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import src.config.settings as generation_settings
import src.config.paths as paths
import src.pipeline.run_pipeline as run_pipeline

In [ ]:
def build_checkbox_tree(root_path, label):
    items = []
    checked = {}

    def walk(path, indent=0):
        if os.path.isdir(path):
            name = os.path.basename(path)
            # Directory shown as a plain label, not a checkbox
            lbl = widgets.Label(value=("  " * indent) + f"📁 {name}",
                                layout=widgets.Layout(width='300px'))
            items.append(lbl)
            for child in sorted(os.listdir(path)):
                walk(os.path.join(path, child), indent + 1)
        else:
            name = os.path.basename(path)
            cb = widgets.Checkbox(value=True, description=("  " * indent) + name,
                                  layout=widgets.Layout(width='300px'))
            checked[path] = cb
            items.append(cb)

    walk(root_path)

    master = widgets.Checkbox(value=True, description=f"✓ {label} (all)",
                              layout=widgets.Layout(width='300px'))

    def on_master_change(change):
        for cb in checked.values():
            cb.value = change['new']

    master.observe(on_master_change, names='value')

    return widgets.VBox([master, *items]), checked

cc_tree, checked_CC = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'cranial-caudal', "Cranial-Caudal")
ml_tree, checked_ML = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'medial-lateral', "Medial-Lateral")
pd_tree, checked_PD = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'proximal-distal', "Proximal-Distal")

tab2 = widgets.HBox([cc_tree, ml_tree, pd_tree])

In [ ]:
printer_box = widgets.Dropdown(options=["BMF", "Formlabs"], value="BMF", description="Printer:")
method_box = widgets.Dropdown(options=["Tension", "Compression"], value="Tension", description="Method:")

medial_lateral_enabled = widgets.Checkbox(value=True, description="Medial-Lateral")
cranial_caudal_enabled = widgets.Checkbox(value=True, description="Cranial-Caudal")
proximal_distal_enabled = widgets.Checkbox(value=True, description="Proximal-Distal")
median_graph_enabled = widgets.Checkbox(value=True, description="Median Graph")

orientation_row = widgets.HBox([medial_lateral_enabled, cranial_caudal_enabled, proximal_distal_enabled])

all_checked = {**checked_CC, **checked_ML, **checked_PD}
unselected = [path for path, cb in all_checked.items() if not cb.value]

orientations = {
    'medial-lateral': medial_lateral_enabled,
    'cranial-caudal': cranial_caudal_enabled,
    'proximal-distal': proximal_distal_enabled
}

current_analysis = generation_settings.Analysis(median_graph_enabled, printer_box, method_box, orientations, unselected)

generate_btn = widgets.Button(description="Generate Graph", button_style="primary")
plot_output = widgets.Output()  # ← this is your canvas replacement

def on_generate(b):
    generate_btn.disabled = True
    generate_btn.description = "Running..."
    with plot_output:
        clear_output(wait=True)
        # call your run_pipeline here, it renders matplotlib into this output
        run_pipeline.generate_graph(current_analysis)
    generate_btn.disabled = False
    generate_btn.description = "Generate Graph"

generate_btn.on_click(on_generate)

tab1 = widgets.VBox([printer_box, method_box, orientation_row, median_graph_enabled, generate_btn, plot_output])

In [ ]:
tabs = widgets.Tab(children=[tab1, tab2])
tabs.set_title(0, "Analysis")
tabs.set_title(1, "Files")

display(tabs)